# 기초 B2 · 세 집단 이상

이 노트북은 구글 **Colab**에서 바로 실행됩니다. 위에서부터 각 셀을 **Shift+Enter** 로 실행하세요. 설치는 없고, 구글 계정만 있으면 됩니다.

📖 본문 학습 페이지: [기초 B2 · 세 집단 이상](https://grow.minds.kr/textbooks/css-methods/causal/book/b2-세-집단-이상.html)

## 1. 준비

In [ ]:
# 이 책의 데이터·코드를 코랩으로 내려받습니다(처음 한 번, 수 초).
!git clone -q https://github.com/dataminds/css-methods-causal-code.git
%cd css-methods-causal-code

In [ ]:
import pandas as pd, numpy as np
from scipy import stats

def load(name, clean=True):
    df = pd.read_csv(f"data/journey_{name}.csv")
    return df[df.attn_1 == 1] if clean and "attn_1" in df else df

def ols(y, X):                      # 절편 포함 최소제곱 → (계수, 표준오차, p, R^2)
    y = np.asarray(y, float)
    X1 = np.column_stack([np.ones(len(y))] + [np.asarray(x, float) for x in X])
    b, *_ = np.linalg.lstsq(X1, y, rcond=None)
    resid = y - X1 @ b
    n, k = X1.shape
    se = np.sqrt(np.diag(resid @ resid / (n - k) * np.linalg.inv(X1.T @ X1)))
    p = 2 * stats.t.sf(np.abs(b / se), n - k)
    r2 = 1 - (resid @ resid) / ((y - y.mean()) @ (y - y.mean()))
    return b, se, p, r2

def cohen_d(a, b):
    sp = np.sqrt(((len(a)-1)*a.std(ddof=1)**2 + (len(b)-1)*b.std(ddof=1)**2) / (len(a)+len(b)-2))
    return (a.mean() - b.mean()) / sp

def cronbach(items):
    items = np.asarray(items, float); k = items.shape[1]
    return k/(k-1) * (1 - items.var(axis=0, ddof=1).sum() / items.sum(axis=1).var(ddof=1))

print("준비 끝. 데이터와 도우미 함수를 불러왔습니다.")


## 2. 일원배치 분산분석
요인판의 네 셀을 네 집단으로 보고 *F*를 냅니다. *F*는 집단 사이 흩어짐을 집단 안 흩어짐으로 나눈 값입니다.

In [ ]:
fac = load("fac")
grp = [fac[(fac.elem == e) & (fac.frame == f)].mil_t2.values
       for e in (0, 1) for f in (0, 1)]
print([len(x) for x in grp], [round(x.mean(), 2) for x in grp])
F, p = stats.f_oneway(*grp)
print(round(F, 3), round(p, 4))
# [110, 106, 109, 105] [4.9, 4.81, 4.95, 5.2] / 1.728 0.1605

## 3. ⭐⭐ 전체는 유의하지 않은데 쌍별로는
네 줄이 이 단위의 전부입니다. 쌍이 여섯이면 그중 가장 작은 *p*는 그쯤 나옵니다.

In [ ]:
pairs = [(i, j) for i in range(4) for j in range(i + 1, 4)]
ps = [stats.ttest_ind(grp[i], grp[j]).pvalue for i, j in pairs]
print(len(ps), sum(q < .05 for q in ps), round(min(ps), 4))
print(round(.05 / len(ps), 4), sum(q < .05 / len(ps) for q in ps))
# 6 1 0.0308  ← 쌍별로 보면 하나가 유의하다
# 0.0083 0     ← 본페로니 문턱을 대면 0개가 남는다

## 4. 직접 바꿔 보기
위 셀의 숫자(씨앗 73, 표본 크기, 제외 기준 등)를 바꿔 다시 실행해 보세요. 결과가 어떻게 달라지나요?

> **검증 로그(부록 B)**: 무엇을 바꿨고, 무엇이 나왔고, 예상과 같았는지 한 문단으로 적어 두세요. 실행이 아니라 검증이 이 책의 핵심입니다.